# Support Ops Env — GRPO Training on Kaggle

**OpenEnv Hackathon** | [HF Space](https://huggingface.co/spaces/raj23211/support-ops-env)

### Before you run

1. **Settings → Accelerator → GPU** (T4/P100; same VRAM discipline as Colab).
2. **Settings → Internet → On** (required for `pip install`, Hugging Face Hub, and calls to your Space).
3. **Secrets:** add `HF_TOKEN` in **Add-ons → Secrets** (or the notebook **Input** menu), attach it to this notebook, then run the config cell.

Kaggle GPU sessions are time-limited (~**12 hours** with GPU). Save checkpoints and `reward_log.csv` under `/kaggle/working` (default below) and **download** before the session ends.


### Notebook from GitHub vs. Internet

**Add notebook from GitHub** (or a GitHub link) only copies the `.ipynb` into Kaggle. It does **not** turn on outbound internet. Pip and `socket.gethostbyname` use **Kaggle’s VM network**, controlled by **Session options → Internet** (must be **ON**).

Same stack as Colab: **TRL GRPO**, optional **4-bit QLoRA** on smaller GPUs, optional **Unsloth** path.


## 1. Install Dependencies

Requires **Internet enabled** on this notebook (Kaggle: **Settings → Internet → On**). Installs TRL stack + `openenv-support-ops` from your HF Space git URL.

If `pip` prints **`Temporary failure in name resolution`** or **`No matching distribution found for trl … (from versions: none)`**, that is almost always **no DNS / no outbound internet** — not a missing TRL version. Turn Internet on, restart the session, run the **network check** cell below, then retry install.


In [ ]:
# Network check — run before pip. Fails fast if this session has no DNS / outbound internet.
import socket
import urllib.request

hosts = ("pypi.org", "files.pythonhosted.org", "huggingface.co")
try:
    for host in hosts:
        socket.gethostbyname(host)
        print("resolve OK:", host)
    _ = urllib.request.urlopen("https://pypi.org/simple/pip/", timeout=20).read(64)
    print("HTTPS to PyPI OK — proceed to the next cell.")
except socket.gaierror as e:
    raise SystemExit(
        "DNS failed (gaierror / Errno -3). Importing this notebook from GitHub only copies the file; "
        "it does NOT enable internet. Your code runs on Kaggle's VM.\n\n"
        "Fix: Settings / Session options → Internet → ON → Save → Restart session, then re-run this cell.\n\n"
        "If the kernel was created via API, open the kernel on kaggle.com and confirm Internet is enabled.\n"
        "Detail: " + repr(e)
    ) from e
except OSError as e:
    raise SystemExit(
        "Could not reach PyPI over HTTPS. Turn Internet ON, then retry.\nDetail: " + repr(e)
    ) from e


### During `pip install`

**Do not press Stop / Ctrl+C** while pip prints `Retrying ... name resolution`. Cancelling leaves **no `trl`** (and you get `ModuleNotFoundError`). Let retries finish, or fix **Internet** first and re-run from the network check cell.


In [ ]:
# Qwen3-4B-Instruct-2507 requires transformers >= 5.0.
# On T4 we use 4-bit NF4 via bitsandbytes, and skip vLLM (doesn't fit alongside a 4B colocate KV cache).
# Pin a few shared runtime packages so Colab/Kaggle preinstalls remain compatible.
# These are not DriftShield requirements; they keep the surrounding notebook image stable.
%pip install -q --no-cache-dir \
    "cachetools<7" \
    "rich<14" \
    "fsspec==2025.3.0" \
    "requests==2.32.4" \
    "numpy<2.2" \
    "pandas<2.4" \
    "cryptography<44" \
    "docutils<0.22" \
    "opentelemetry-api==1.38.0" \
    "starlette<1.0.0" \
    "websockets<16.0.0" \
    "jedi>=0.16,<0.20" \
    "transformers>=5.0" \
    "trl>=0.29.0" \
    "peft>=0.13.0" \
    "accelerate>=0.30" \
    "bitsandbytes>=0.43.0" \
    "datasets" \
    "huggingface_hub>=0.20.0" \
    "matplotlib" \
    "numpy"

# Install our env package (client + models + tasks + training utils).
# Force-reinstall so the runtime picks up the latest pushed train.py instead of
# reusing an older editable/wheel install from a previous session.
%pip uninstall -y openenv-support-ops >/dev/null 2>&1 || true
%pip install -q --no-cache-dir --force-reinstall --no-deps \
    "openenv-support-ops @ git+https://huggingface.co/spaces/raj23211/support-ops-env@main"
%pip install -q --no-cache-dir --force-reinstall --no-deps "openenv-core==0.2.3"

# Imports run in the *next* cell so Ctrl+C on pip does not land on a confusing import traceback.


In [ ]:
# Verify installs (run after pip cell finishes — do not press Stop while pip is retrying).
try:
    import transformers
    import trl
    import peft
except ModuleNotFoundError as e:
    raise SystemExit(
        "Missing package after pip. Common causes: (1) You pressed Stop/Ctrl+C while pip was "
        "retrying DNS errors — packages never installed. (2) Internet still off — fix Settings → Internet, "
        "re-run the network check cell, then re-run the pip cell to completion.\n"
        f"Import error: {e}"
    ) from e

print(f"transformers : {transformers.__version__}")
print(f"trl          : {trl.__version__}")
print(f"peft         : {peft.__version__}")


## 1b. (Optional) Install Unsloth

Only run this cell if you set `USE_UNSLOTH = True` in Section 2. Unsloth gives ~1.5× faster training and ~50% less VRAM on Qwen3.5-4B (official docs). Skip otherwise — the stable path above is enough.

In [ ]:
# Only needed when USE_UNSLOTH=True. Safe to re-run (idempotent).
!pip install -q --upgrade --no-cache-dir unsloth unsloth_zoo
import unsloth
print(f"unsloth : {unsloth.__version__}")

## 2. Configuration

Set `HF_TOKEN` via **Kaggle Secrets** (`HF_TOKEN` key). On Colab, `google.colab.userdata` still works if you reuse this file there.


In [ ]:
import os
from pathlib import Path


def _load_hf_token():
    # Kaggle: Add-ons → Secrets → HF_TOKEN, attach to notebook.
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore

        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    # Colab (if you open the same notebook there)
    try:
        from google.colab import userdata  # type: ignore

        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")


_tok = _load_hf_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
else:
    print("WARNING: HF_TOKEN not set. Add secret HF_TOKEN (Kaggle) or Colab userdata.")

# Persist artifacts here so you can download them from the Output tab.
WORK_ROOT = Path("/kaggle/working") if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else Path(".")

ENV_URL         = "https://raj23211-support-ops-env.hf.space"
USE_UNSLOTH     = False       # Flip True for Unsloth Qwen3.5-4B (bf16 LoRA, ~10 GB, ~1.5x faster)

if USE_UNSLOTH:
    MODEL_ID     = "unsloth/Qwen3.5-4B"
    LOAD_IN_4BIT = False      # Unsloth docs: QLoRA not recommended for Qwen3.5
    USE_VLLM     = False      # Unsloth uses its own generation (fast_inference=False)
    HUB_REPO     = "raj23211/support-ops-agent-qwen35-4b-unsloth"
else:
    MODEL_ID     = "Qwen/Qwen3-4B-Instruct-2507"
    LOAD_IN_4BIT = True       # NF4 + bf16 compute fits T4
    USE_VLLM     = False      # True only on A100/L4 24GB+
    HUB_REPO     = "raj23211/support-ops-agent-qwen3-4b-instruct"

NUM_EPISODES    = 50
NUM_GENERATIONS = 4           # per prompt — T4 handles 4
MAX_TURNS       = 16          # DriftShield needs room to recover from runtime errors
DIFFICULTY      = "driftshield_easy"  # curriculum: driftshield | driftshield_easy | easy | medium | hard | expert | all

print(f"Environment : {ENV_URL}")
print(f"Model       : {MODEL_ID}")
print(f"Episodes    : {NUM_EPISODES}")
print(f"Generations : {NUM_GENERATIONS}")
print(f"vLLM        : {USE_VLLM}")
print(f"4-bit NF4   : {LOAD_IN_4BIT}")
print(f"Artifact root: {WORK_ROOT}")


## 3. Smoke Test — Verify Environment Connectivity

Connect to the HF Space, reset the environment (picks a task), and execute one investigative tool call to confirm round-trip works.

In [ ]:
from support_ops_env import SupportOpsAction, SupportOpsEnv, TASK_IDS

print(f"Connecting to {ENV_URL} ...")

env = SupportOpsEnv(base_url=ENV_URL).sync()

try:
    reset_result = env.reset(task_id=TASK_IDS[0])
    obs = reset_result.observation
    print("Connected!\n")
    print(f"Task       : {obs.task_title}")
    print(f"Objective  : {obs.objective[:200]}...\n")

    step = env.step(
        SupportOpsAction(
            assistant_message="Listing inbox to find the primary case.",
            tool_calls=[{"name": "inbox.list_cases", "args": {}}],
            answer=None,
        )
    )
    print(f"--- smoke test step (reward={step.reward:.2f}) ---")
    for tr in step.observation.tool_results[-1:]:
        print(f"{tr.name} ok={tr.ok}\n{str(tr.result)[:400]}")

finally:
    if hasattr(env, 'close'):
        env.close()

print("\nSmoke test passed. Environment is ready for training.")

## 4. Import Training Utilities from Package

All training logic (system prompt, rollout function, reward functions, helpers) is imported directly from `support_ops_env.train` — the same code used for H100 training. No duplication.

In [ ]:
import csv
import logging
from datetime import datetime
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

from support_ops_env import get_training_utils

tu = get_training_utils()
SYSTEM_PROMPT         = tu["SYSTEM_PROMPT"]
rollout_once          = tu["rollout_once"]
reward_total          = tu["reward_total"]
reward_fields         = tu["reward_fields"]
reward_reply          = tu["reward_reply"]
reward_grounding      = tu["reward_grounding"]
plot_rewards          = tu["plot_rewards"]
patch_trl_vllm_compat = tu["patch_trl_vllm_compat"]

patch_trl_vllm_compat()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

print("System prompt (first 200 chars):")
print(SYSTEM_PROMPT[:200])
print("...")
print("\nImported: rollout_once, reward_total, reward_fields, reward_reply, reward_grounding.")

## 5. GRPO Training Setup

Configure tokenizer, environment connection, reward logging, and the GRPO trainer — all using functions imported from the package.

In [ ]:
if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    loaded_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=4096,
        load_in_4bit=False,
        load_in_16bit=True,
        full_finetuning=False,
        fast_inference=False,   # required for Qwen3.5 GRPO (Unsloth docs)
    )
    loaded_model = FastLanguageModel.get_peft_model(
        loaded_model,
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        max_seq_length=4096,
    )
else:
    loaded_model = None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

env = SupportOpsEnv(base_url=ENV_URL).sync()
dataset = Dataset.from_dict(
    {"prompt": ["Triage and resolve this support operations case."] * NUM_EPISODES}
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = WORK_ROOT / f"outputs/support-ops-grpo-{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)

reward_log_path = output_dir / "reward_log.csv"
episode_counter = [0]
all_totals = []

with open(reward_log_path, "w", newline="") as fh:
    csv.writer(fh).writerow([
        "episode", "task_id", "total_reward", "investigation", "routing", "reply_quality", "groundedness", "submission", "penalty_total", "parse_ok_ratio", "timestamp",
    ])

DIAGNOSTIC_EVERY = 5  # print a schema-health line every N episodes

def log_episode(ep):
    episode_counter[0] += 1
    total_r = float(ep["total_reward"])
    all_totals.append(total_r)
    parse_ok = float(ep.get("parse_ok_ratio", 0.0))
    with open(reward_log_path, "a", newline="") as fh:
        csv.writer(fh).writerow([
            episode_counter[0], ep.get("task_id", ""), total_r,
            round(float(ep.get("investigation_reward", 0.0)), 4),
            round(float(ep.get("routing_reward", 0.0)), 4),
            round(float(ep.get("reply_reward", 0.0)), 4),
            round(float(ep.get("grounding_reward", 0.0)), 4),
            round(float(ep.get("submission_reward", 0.0)), 4),
            round(float(ep.get("penalty_total", 0.0)), 4),
            round(parse_ok, 4), datetime.now().isoformat(),
        ])
    n = episode_counter[0]
    last10 = all_totals[-10:]
    logger.info(
        f"Episode {n}: reward={total_r:+.2f} | "
        f"mean={sum(all_totals)/len(all_totals):+.2f}, mean(10)={sum(last10)/len(last10):+.2f}"
    )
    # Every-N-episodes schema-health diagnostic.
    if n % DIAGNOSTIC_EVERY == 0:
        attempts = int(ep.get("parse_attempts", 0))
        failures = int(ep.get("parse_failures", 0))
        err = ep.get("last_parse_error") or "(none)"
        raw = (ep.get("last_raw_completion") or "").replace("\n", " ")[:160]
        logger.info(
            f"[diagnostic ep={n}] parse_ok={parse_ok:.2f} ({attempts-failures}/{attempts}) "
            f"| last_err={err} | last_raw[:160]={raw!r}"
        )
        if parse_ok < 0.5:
            logger.warning(
                f"[diagnostic ep={n}] parse_ok_ratio<0.5 - model is producing invalid actions. "
                "Check SYSTEM_PROMPT few-shot, max_completion_length, or use a stronger base."
            )

from support_ops_env import get_curriculum_task_ids

curriculum = get_curriculum_task_ids(DIFFICULTY)
print(f"Curriculum [{DIFFICULTY}] -> {curriculum}")
_task_cursor = [0]

def _next_task_id():
    tid = curriculum[_task_cursor[0] % len(curriculum)]
    _task_cursor[0] += 1
    return tid

def rollout_func(prompts, trainer):
    results = {k: [] for k in [
        "prompt_ids", "completion_ids", "logprobs",
        "total_reward", "field_reward", "reply_reward", "grounding_reward",
    ]}
    for _ in prompts:
        ep = rollout_once(trainer, env, tokenizer, SYSTEM_PROMPT, MAX_TURNS,
                          task_id=_next_task_id())
        for k in results:
            results[k].append(ep[k])
        log_episode(ep)
    return results

print(f"Training setup complete. Output: {output_dir}")

In [ ]:
import torch

grpo_kwargs = dict(
    output_dir=str(output_dir),
    num_train_epochs=1,
    learning_rate=2e-6,
    gradient_accumulation_steps=4,
    per_device_train_batch_size=NUM_GENERATIONS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=512,
    logging_steps=1,
    save_strategy="steps",
    save_steps=10,
    temperature=0.7,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_total_limit=3,
)

if USE_VLLM and not USE_UNSLOTH:
    grpo_kwargs.update(
        use_vllm=True,
        vllm_mode="colocate",
        vllm_gpu_memory_utilization=0.5,
    )

if LOAD_IN_4BIT and not USE_UNSLOTH:
    from transformers import BitsAndBytesConfig
    # NF4 weights + bf16 compute. LoRA adapters stay in bf16 (QLoRA).
    grpo_kwargs["model_init_kwargs"] = {
        "torch_dtype": torch.bfloat16,
        "quantization_config": BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        ),
    }

grpo_config = GRPOConfig(**grpo_kwargs)

if USE_UNSLOTH:
    # Unsloth already attached LoRA via get_peft_model above.
    model_arg = loaded_model
    peft_config = None
else:
    from peft import LoraConfig
    model_arg = MODEL_ID
    peft_config = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        bias="none", task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )

trainer = GRPOTrainer(
    model=model_arg,
    processing_class=tokenizer,
    reward_funcs=[reward_total, reward_fields, reward_reply, reward_grounding],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
    peft_config=peft_config,
)

print("GRPOTrainer initialized")
print(f"  Unsloth       : {USE_UNSLOTH}")
print(f"  vLLM          : {USE_VLLM}")
print(f"  4-bit NF4     : {LOAD_IN_4BIT}")
print(f"  VRAM now      : {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 5b. Schema sanity probe (run BEFORE trainer.train())

Stops you from burning a 50-episode run on a model that can't speak the action schema.

**Pass criteria:** at least one of the 3 probes parses to a valid `SupportOpsAction` with non-empty `tool_calls`.
If they all fail, the model is producing prose / markdown / `<think>` blocks instead of strict JSON, and GRPO will only see flat -1.0 rewards. Add a few-shot example to `SYSTEM_PROMPT` (already done in the latest train.py) or switch to a stronger base.


In [ ]:
import json as _json
from support_ops_env import SupportOpsAction
from support_ops_env.train import SYSTEM_PROMPT, format_observation

env_local = SupportOpsEnv(base_url=ENV_URL).sync()
PROBES = ["ds_prompt_injection_access", "ds_schema_drift_refund", "ds_lying_tool_gdpr"]
results = []
for tid in PROBES:
    obs = env_local.reset(task_id=tid).observation
    text = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": format_observation(obs)}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(trainer.model.device)
    out = trainer.model.generate(**inputs, max_new_tokens=512, do_sample=False)
    raw = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        action = SupportOpsAction(**_json.loads(clean))
        results.append((tid, "OK", f"{len(action.tool_calls)} tool_calls -> {[t.name for t in action.tool_calls[:3]]}"))
    except Exception as e:
        results.append((tid, "FAIL", f"{type(e).__name__}: {str(e)[:120]} | raw[:120]={raw[:120]!r}"))

print("=" * 70)
for tid, status, info in results:
    print(f"{status:4s}  {tid:32s}  {info}")
print("=" * 70)
ok_count = sum(1 for _, s, _ in results if s == "OK")
if ok_count == 0:
    print("ALL FAILED -> do NOT proceed to trainer.train(). Add few-shot to SYSTEM_PROMPT or use a stronger base.")
elif ok_count < 3:
    print(f"{ok_count}/3 OK -> training will work but expect noisy early episodes.")
else:
    print("3/3 OK -> safe to call trainer.train().")


## 6. Train!

Launch GRPO training. Each episode: `reset` → agent investigates via tool calls → agent drafts a reply and submits → deterministic grader scores → GRPO gradient update.

In [ ]:
print("Starting GRPO training...")
print(f"  Model       : {MODEL_ID}")
print(f"  Episodes    : {NUM_EPISODES}")
print(f"  Generations : {NUM_GENERATIONS}")
print(f"  Environment : {ENV_URL}")
print()

try:
    trainer.train()
finally:
    if hasattr(env, "close"):
        env.close()

trainer.save_model(str(output_dir))
print(f"\nModel saved to {output_dir}")

## 7. Reward Curves

Visualize training progress using `plot_rewards` from the package.

In [ ]:
plot_rewards(reward_log_path, output_dir / "reward_curve.png")

import matplotlib.pyplot as plt
from matplotlib.image import imread

img = imread(str(output_dir / "reward_curve.png"))
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(img)
ax.axis("off")
plt.show()

print(f"Reward curve saved to {output_dir / 'reward_curve.png'}")

## 8. Push to Hub (Optional)

Upload the trained LoRA adapter to Hugging Face Hub.

In [ ]:
# Uncomment to push:
# trainer.push_to_hub(repo_id=HUB_REPO)
# print(f"Model pushed to https://huggingface.co/{HUB_REPO}")